# 07 - Extraccion final para el equipo de visualizacion

Toma las columnas objetivo que el diagnostico de `06_viabilidad_columnas.ipynb`
confirmo viables y genera la version limpia de datos para el dashboard de VcM.

**Principio: honestidad de cobertura, sin mezclar anios con distinta
disponibilidad.** Por eso la salida son **dos tablas separadas**, no una sola
con huecos:

- **Tabla A - `dashboard_participantes_2024_2025`** (dataset rico): solo
  2024-2025, con participantes desagregados (estudiantes, academicos/docentes,
  titulados), organizaciones de la sociedad civil, competencias (sello y
  genericas) y catedra/asignatura donde exista. Es el dataset principal para las
  metricas de participantes que pidio la contraparte.
- **Tabla B - `dashboard_historico_2022_2025`** (serie comparable): todos los
  anios 2022-2025, pero solo columnas comparables en todo el periodo
  (identificador, facultad, carrera, instrumento, anio, estado, competencia
  sello y total agregado de asistentes/participantes). Sirve para graficos de
  evolucion del numero de iniciativas y su distribucion, sin prometer una
  desagregacion que los anios viejos no tienen.

**Reglas aplicadas**

- Reutiliza la busqueda flexible de columnas validada en el notebook 06, con los
  mismos `excludes` que evitaron los falsos positivos de `Sello Institucional` y
  del conteo de alumnos de catedra.
- Universo = iniciativas reales (fila con identificador), no plantillas vacias.
- Deduplica los archivos redundantes de 2025 (copias con mojibake `Ã` / sufijo `(1)`).
- Conteos como numericos con `NaN` donde no hay dato; nunca `0` por defecto.
- Multivalor (competencias) con separador `; ` consistente.
- Columnas de linaje `_archivo_origen` y `_anio` para trazabilidad.
- **Emprendedores / empleadores NO se incluye**: el diagnostico confirmo que no
  existe esa columna en ninguna fuente. Se documenta su ausencia, no se fuerza.

Esto solo lee `data/raw` y escribe en `data/clean`. No modifica `src/`.

In [1]:
import csv
import re
import unicodedata
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")  # validaciones de openpyxl no soportadas; no afectan la lectura
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

RAW = Path("..") / "data" / "raw"
CLEAN = Path("..") / "data" / "clean"
assert RAW.exists(), f"No existe {RAW.resolve()}"
print("Origen:", RAW.resolve())
print("Destino:", CLEAN.resolve())

Origen: C:\Users\ignac\vscode projects\visualizacion de datos\data\raw
Destino: C:\Users\ignac\vscode projects\visualizacion de datos\data\clean


## 1. Logica de busqueda flexible (reutilizada del notebook 06)

Mismas funciones de normalizacion, mapeo de columnas por palabras clave,
seleccion de hoja principal, filtro de iniciativas reales e inferencia de anio.

In [2]:
def norm(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", s.lower()).strip()


def match_columns(cols, spec):
    hits = []
    for c in cols:
        nc = norm(c)
        if any(k in nc for k in spec["any"]) and not any(k in nc for k in spec.get("not", [])):
            hits.append(c)
    if not hits and spec.get("fallback_any"):
        hits = [c for c in cols if any(k in norm(c) for k in spec["fallback_any"])]
    return hits


def pick_main_sheet(sheets):
    for s in sheets:
        if s.lower().startswith("iniciativas"):
            return s
    for cand in ("PLANILLA", "Planilla"):
        if cand in sheets:
            return cand
    return sheets[0]


def is_populated(series):
    s = series.astype("string").str.strip()
    return s.notna() & (s != "") & (s.str.lower() != "nan")


def iniciativa_mask(df):
    m = np.zeros(len(df), dtype=bool)
    for c in df.columns:
        nc = norm(c)
        if ("codigo" in nc) or ("nombre de la iniciativa" in nc) or \
           ("nombre de iniciativa" in nc) or (nc == "nombre"):
            m = m | is_populated(df[c]).to_numpy()
    return m


def infer_year(path, df):
    m = re.search(r"(20\d{2})", path.stem)
    if m:
        return int(m.group(1))
    for c in df.columns:
        nc = norm(c)
        if "fecha" in nc and any(k in nc for k in ("termin", "inicio", "finaliza")):
            years = pd.to_datetime(df[c], errors="coerce").dt.year.dropna()
            if len(years):
                return int(years.mode().iloc[0])
    return None


def detect_instrumento(name):
    n = norm(name)
    if ("vedp" in n) or ("entorno disciplinar" in n):
        return "VEDP"
    if ("titulados" in n) or re.search(r"\bvct\b", n):
        return "VT"
    if "centraliz" in n:
        return "CENTRALIZADAS"
    if "utg" in n:
        return "UTG"
    if "fcr" in n:
        return "FCR"
    if ("ext" in n) or ("extension" in n):
        return "EXTENSION"
    return "OTRO"

## 2. Especificaciones de columnas de salida y extractores

Cada columna de salida se define con las mismas palabras clave del diagnostico.
Notar los filtros negativos:

- `estudiantes` excluye `catedra`/`asignatura` (no cuenta alumnos de la catedra).
- `organizaciones` excluye `sello`, `competencia` y `persona`.
- `catedra_asignatura` excluye `estudiante`, `semestre` y `codigo` (queremos el nombre).
- `competencia_sello` excluye `generica`.
- `dominios_disciplinares` (agregado para el KPI 1 del dashboard): existe en la
  fuente solo desde 2024 y es un concepto distinto de las areas genericas de
  2022-2023, por eso no se mapea hacia atras. El formato 2024 separa items con
  `., ` y el 2025 con `;`; NO se divide por coma simple porque las frases tienen
  comas internas (una celda con items separados solo por coma queda como un
  valor unico). `No aplica` se trata como sin dato.
- `cantidad_act_planificadas` / `cantidad_act_ejecutadas` (base del KPI I19):
  planificadas existe en todos los anios; ejecutadas solo en el formato
  2022-2023 (dejo de capturarse despues). Celdas sucias tipo `2; 3` se resuelven
  tomando el primer numero.
- `evidencia` (normalizada a SI/NO): 2022-2023 se deriva de `Estado de
  Evidencia` (COMPLETO/INCOMPLETO -> SI, SIN EVIDENCIA -> NO); 2024-2025 traen
  el campo SI/NO directo.
- `modalidad` (filtro obligatorio del dashboard): existe en todos los formatos
  con variantes de forma (PRESENCIAL/ONLINE/HIBRIDO vs Presencial/Online/
  Hibrida); se normaliza a Presencial / Online / Hibrida.
- `semestre` (filtro obligatorio): existe en todos los formatos como `Semestre
  Ejecucion` (PRIMERO/SEGUNDO/ANUAL) o `Semestre de ejecucion de la iniciativa`
  (Primer/Segundo semestre); se normaliza a 1S / 2S / Anual. Se excluye
  `Semestre de la catedra`, que es el semestre de la asignatura asociada.

Extractores:

- `get_text`: primer valor de texto, `NaN` donde vacio.
- `get_multi`: normaliza separadores multivalor a `; `. Los formatos usan
  separadores distintos (`;` en 2024, `,` en 2025), asi que el sello se divide
  por ambos y la generica respeta las comillas estilo CSV de 2025 para no
  fragmentar las frases largas (que llevan comas internas).
- `get_count`: suma numerica por fila de las columnas mapeadas (combina los
  conteos por sexo M/F/Otro de 2025); `NaN` si todas son `NaN`, nunca `0` por defecto.

In [3]:
SP_CODIGO = {"any": ["codigo"], "not": []}
SP_NOMBRE = {"any": ["nombre de la iniciativa", "nombre de iniciativa"], "not": [], "fallback_any": ["nombre"]}
SP_FACULTAD = {"any": ["facultad"], "not": []}
SP_CARRERA = {"any": ["carrera"], "not": []}
SP_ESTADO = {"any": ["estado sisav"], "not": []}
SP_ESTUD = {"any": ["estudiante"], "not": ["catedra", "asignatura"]}
SP_ACAD = {"any": ["docente", "academico"], "not": []}
SP_TIT = {"any": ["titulado"], "not": []}
SP_ORG = {"any": ["institucion", "sociedad civil", "organizacion"], "not": ["sello", "competencia", "estudiante", "persona"]}
SP_SELLO = {"any": ["sello"], "not": ["generica"]}
SP_GEN = {"any": ["generica"], "not": []}
SP_CAT = {"any": ["asignatura", "catedra"], "not": ["estudiante", "semestre", "codigo"]}
SP_DOM = {"any": ["dominios disciplinares"], "not": []}
SP_TOTAL = {"any": ["total asistentes", "total de participantes"], "not": []}
SP_PLAN = {"any": ["act planificadas", "actividades planificadas"], "not": ["nombre"]}
SP_EJEC = {"any": ["act eje", "actividades ejecutadas"], "not": ["nombre"]}
SP_EVID = {"any": ["evidencia"], "not": []}
SP_MODAL = {"any": ["modalidad"], "not": []}
SP_SEM = {"any": ["semestre ejecucion", "semestre de ejecucion"], "not": ["catedra"]}

MULTI_SEP = "; "


def _first_col(df, spec, prefer=None):
    cols = match_columns(df.columns, spec)
    if prefer:
        pc = [c for c in cols if prefer in norm(c)]
        if pc:
            cols = pc
    return cols[0] if cols else None


def get_text(df, spec, prefer=None):
    c = _first_col(df, spec, prefer)
    if c is None:
        return pd.Series([pd.NA] * len(df), dtype="string")
    s = df[c].astype("string").str.strip()
    return s.where(is_populated(df[c]), other=pd.NA)


def get_multi(df, spec, mode="labels"):
    """Normaliza un campo multivalor al separador `; `.

    Los separadores de origen son inconsistentes entre formatos:
    - `sello` (mode='labels'): etiquetas cortas de vocabulario controlado, sin
      comas internas; separadas por `;` (2024) o `,` (2025). Se divide por ambos.
    - `generica` (mode='phrases'): frases largas que contienen comas internas;
      separadas por `;` (2024) o por `,` entre etiquetas entrecomilladas estilo
      CSV (2025). Se divide por `;` siempre y, solo si hay comillas, por coma
      respetando las comillas. Asi no se fragmentan las frases.
    - `dominios` (mode='dominios'): frases largas con comas internas que NO se
      dividen por coma simple; los separadores reales son `;`, salto de linea y
      el patron `., ` (punto seguido de coma, tipico del formato 2024). Se
      limpian los puntos finales y `No aplica` se trata como sin dato.
    """
    c = _first_col(df, spec)
    if c is None:
        return pd.Series([pd.NA] * len(df), dtype="string")

    sep = r"[;\n]+|\.\s*," if mode == "dominios" else r"[;\n]+"

    def f(v):
        if pd.isna(v):
            return pd.NA
        s = str(v).strip()
        if s == "" or s.lower() == "nan":
            return pd.NA
        out = []
        for chunk in re.split(sep, s):
            chunk = chunk.strip()
            if not chunk:
                continue
            if mode == "labels":
                partes = chunk.split(",")
            elif mode == "dominios":
                partes = [chunk]
            elif '"' in chunk:
                # coma respetando comillas; skipinitialspace para las comillas
                # precedidas de espacio (formato real de 2025)
                partes = next(csv.reader([chunk], skipinitialspace=True))
            else:
                partes = [chunk]
            for p in partes:
                p = re.sub(r"\s+", " ", p.strip().strip('"').strip())
                if mode == "dominios":
                    p = p.rstrip(".").strip()
                    if p.lower() in ("no aplica", "n/a"):
                        continue
                if p and p.lower() != "nan" and p not in out:
                    out.append(p)
        return MULTI_SEP.join(out) if out else pd.NA

    return df[c].map(f).astype("string")


def get_count(df, spec, include_total_exact=False, primer_numero=False):
    cols = match_columns(df.columns, spec)
    if include_total_exact:
        cols = cols + [c for c in df.columns if norm(c) == "total" and c not in cols]
    if not cols:
        return pd.Series([np.nan] * len(df), dtype="float")
    if primer_numero:
        # celdas sucias tipo '2; 3': se toma el primer numero que aparezca
        num = df[cols].apply(lambda s: pd.to_numeric(
            s.astype(str).str.extract(r"(\d+(?:[\.,]\d+)?)", expand=False)
            .str.replace(",", ".", regex=False), errors="coerce"))
    else:
        num = df[cols].apply(pd.to_numeric, errors="coerce")
    return num.sum(axis=1, min_count=1)


def get_modalidad(df):
    """Normaliza modalidad: PRESENCIAL/Presencial -> Presencial, ONLINE/Online ->
    Online, HIBRIDO/HÍBRIDA/Híbrida -> Hibrida, Semipresencial se conserva."""
    c = _first_col(df, SP_MODAL)
    if c is None:
        return pd.Series([pd.NA] * len(df), dtype="string")

    def f(v):
        if pd.isna(v):
            return pd.NA
        nv = norm(v)
        if "semipresencial" in nv:
            return "Semipresencial"
        if "presencial" in nv:
            return "Presencial"
        if ("online" in nv) or ("virtual" in nv):
            return "Online"
        if "hibrid" in nv:
            return "Hibrida"
        return str(v).strip().capitalize() or pd.NA

    return df[c].map(f).astype("string")


def get_semestre(df):
    """Normaliza el semestre de ejecucion: PRIMERO/'Primer semestre' -> 1S,
    SEGUNDO/'Segundo semestre' -> 2S, ANUAL -> Anual. Excluye 'Semestre de la
    catedra', que es el semestre de la asignatura asociada, no el de ejecucion."""
    c = _first_col(df, SP_SEM)
    if c is None:
        return pd.Series([pd.NA] * len(df), dtype="string")

    def f(v):
        if pd.isna(v):
            return pd.NA
        nv = norm(v)
        if "primer" in nv or nv == "1":
            return "1S"
        if "segundo" in nv or nv == "2":
            return "2S"
        if "anual" in nv or "ambos" in nv:
            return "Anual"
        return pd.NA

    return df[c].map(f).astype("string")


def get_evidencia(df):
    """Normaliza el campo de evidencia a SI/NO.

    La fuente cambia de forma: 2022-2023 trae 'Estado de Evidencia'
    (COMPLETO / INCOMPLETO / SIN EVIDENCIA, con valores sucios tipo
    'IN;COMPLETO'), 2024 trae 'Evidencia' (SI/NO) y 2025 'Existe (informe de)
    evidencia' (Si/No). Regla: SI/Si -> SI; NO/No -> NO; cualquier valor que
    contenga 'completo' (COMPLETO, INCOMPLETO y sus variantes sucias) -> SI
    porque existe evidencia aunque este incompleta; 'SIN EVIDENCIA' -> NO.
    """
    c = _first_col(df, SP_EVID)
    if c is None:
        return pd.Series([pd.NA] * len(df), dtype="string")

    def f(v):
        if pd.isna(v):
            return pd.NA
        nv = norm(v)
        if nv == "si":
            return "SI"
        if nv == "no":
            return "NO"
        if "completo" in nv:
            return "SI"
        if "sin evidencia" in nv:
            return "NO"
        return pd.NA

    return df[c].map(f).astype("string")

## 3. Lectura, deduplicacion y exclusion de archivos sin anio

Se descartan las copias redundantes (mojibake `Ã` / sufijo `(n)`). Los archivos
sin anio determinable (nombre sin anio y sin fechas parseables) se **excluyen**
para no inventar un periodo; esto mantiene las cifras coherentes con el
diagnostico. Se listan de forma explicita.

In [4]:
todos = sorted(p for p in RAW.iterdir() if p.suffix.lower() in (".xlsm", ".xlsx"))
analizar = [p for p in todos if ("\u00c3" not in p.name) and not re.search(r"\(\d+\)", p.stem)]

datos, sin_anio = {}, []
for p in analizar:
    hoja = pick_main_sheet(pd.ExcelFile(p).sheet_names)
    df = pd.read_excel(p, sheet_name=hoja).dropna(how="all")
    df = df[iniciativa_mask(df)].reset_index(drop=True)
    anio = infer_year(p, df)
    if anio is None:
        sin_anio.append((p.name, len(df)))
        continue
    datos[p.name] = {"anio": anio, "df": df, "instrumento": detect_instrumento(p.name)}

print("Excluidos por anio no determinable (coherente con notebook 06):")
for n, k in sin_anio:
    print(f"   - {n}  ({k} iniciativas)")
print(f"\nArchivos incorporados: {len(datos)}")
for n, i in sorted(datos.items(), key=lambda kv: (kv[1]["anio"], kv[0])):
    print(f"   {i['anio']}  {i['instrumento']:14s} n={len(i['df']):4d}  {n}")

Excluidos por anio no determinable (coherente con notebook 06):
   - Plantilla Iniciativas Centralizadas.xlsm  (61 iniciativas)

Archivos incorporados: 13
   2022  EXTENSION      n=  71  Plantilla EXT 2022.xlsm
   2022  FCR            n=  15  Plantilla FCR 2022.xlsm
   2022  VEDP           n=  62  Plantilla VEDP 2022.xlsm
   2023  EXTENSION      n=  71  Plantilla EXT 2023.xlsm
   2023  FCR            n=  18  Plantilla FCR 2023.xlsm
   2023  VEDP           n=  68  Plantilla VEDP 2023.xlsm
   2024  EXTENSION      n=  56  Plantilla EXT 2024.xlsm
   2024  UTG            n=  17  Plantilla UTG2024.xlsm
   2024  VEDP           n= 103  Plantilla VEDP 2024.xlsm
   2025  EXTENSION      n= 124  EXTENSIÓN ACADÉMICA 2025.xlsx
   2025  CENTRALIZADAS  n=  45  Plantilla Iniciativas Centralizadas 2025.xlsm
   2025  VEDP           n= 149  VINCULACIÓN CON EL ENTORNO DISCIPLINAR PROFESIONAL 2025.xlsx
   2025  VT             n=  27  VINCULACIÓN CON TITULADOS 2025.xlsx


## 4. Construccion de las dos tablas

In [5]:
def construir(nombre, info, tabla):
    df, anio = info["df"], info["anio"]
    base = pd.DataFrame({
        "codigo": get_text(df, SP_CODIGO),
        "nombre_iniciativa": get_text(df, SP_NOMBRE),
        "facultad": get_text(df, SP_FACULTAD),
        "carrera": get_text(df, SP_CARRERA),
        "instrumento": info["instrumento"],
        "anio": anio,
        "estado_sisav": get_text(df, SP_ESTADO),
        # filtros obligatorios del dashboard, normalizados desde la fuente
        "modalidad": get_modalidad(df),
        "semestre": get_semestre(df),
        # base del KPI I19 y del KPI de evidencia (cobertura desigual por anio,
        # documentada en el diccionario): ejecutadas solo existe en 2022-2023
        "cantidad_act_planificadas": get_count(df, SP_PLAN, primer_numero=True),
        "cantidad_act_ejecutadas": get_count(df, SP_EJEC, primer_numero=True),
        "evidencia": get_evidencia(df),
    })
    if tabla == "A":
        base["n_estudiantes"] = get_count(df, SP_ESTUD)
        base["n_academicos_docentes"] = get_count(df, SP_ACAD)
        base["n_titulados"] = get_count(df, SP_TIT)
        base["n_organizaciones_osc"] = get_count(df, SP_ORG)
        base["competencia_sello"] = get_multi(df, SP_SELLO, mode="labels")
        base["competencia_generica"] = get_multi(df, SP_GEN, mode="phrases")
        base["dominios_disciplinares"] = get_multi(df, SP_DOM, mode="dominios")
        base["catedra_asignatura"] = get_text(df, SP_CAT, prefer="nombre")
    else:
        base["competencia_sello"] = get_multi(df, SP_SELLO, mode="labels")
        base["total_participantes"] = get_count(df, SP_TOTAL, include_total_exact=True)
    base["_archivo_origen"] = nombre
    base["_anio"] = anio
    return base


tabla_A = pd.concat([construir(n, i, "A") for n, i in datos.items() if i["anio"] in (2024, 2025)],
                    ignore_index=True)
tabla_B = pd.concat([construir(n, i, "B") for n, i in datos.items() if i["anio"] in (2022, 2023, 2024, 2025)],
                    ignore_index=True)

print("Tabla A (participantes 2024-2025):", tabla_A.shape)
print("Tabla B (historico 2022-2025):    ", tabla_B.shape)
tabla_A.head(3)

Tabla A (participantes 2024-2025): (521, 22)
Tabla B (historico 2022-2025):     (826, 16)


,codigo,nombre_iniciativa,facultad,carrera,instrumento,anio,estado_sisav,modalidad,semestre,cantidad_act_planificadas,cantidad_act_ejecutadas,evidencia,n_estudiantes,n_academicos_docentes,n_titulados,n_organizaciones_osc,competencia_sello,competencia_generica,dominios_disciplinares,catedra_asignatura,_archivo_origen,_anio
0,2438,Asistencia Técnica Fondo comunidad activa 2025,FCJS,Administración_Pública,EXTENSION,2025,Finalizada,Hibrida,1S,16.0,NaN,SI,0.0,0.0,0.0,5.0,Género; Inclusión,"Competencias genéricas para la ciudadanía, la ...",Gestión organizacional; Gestión de P. Públicas,<NA>,EXTENSIÓN ACADÉMICA 2025.xlsx,2025
1,2537,Difusión carrera de administración pública- UTEM,FCJS,Administración_Pública,EXTENSION,2025,Finalizada,Presencial,1S,1.0,NaN,SI,27.0,0.0,0.0,0.0,Responsabilidad social; Inclusión,"Competencias genéricas para la ciudadanía, la ...",Gestión organizacional pública,GESTIÓN I,EXTENSIÓN ACADÉMICA 2025.xlsx,2025
2,2409,"Entre líneas: Procesos, relevos y traspasos en...",FCCOT,Arquitectura,EXTENSION,2025,Finalizada,Presencial,1S,1.0,NaN,SI,38.0,1.0,0.0,0.0,Género,"Competencias genéricas para la ciudadanía, la ...",Valorar,Teoría e historia de la arquitectura II,EXTENSIÓN ACADÉMICA 2025.xlsx,2025


In [6]:
tabla_B.head(3)

,codigo,nombre_iniciativa,facultad,carrera,instrumento,anio,estado_sisav,modalidad,semestre,cantidad_act_planificadas,cantidad_act_ejecutadas,evidencia,competencia_sello,total_participantes,_archivo_origen,_anio
0,2438,Asistencia Técnica Fondo comunidad activa 2025,FCJS,Administración_Pública,EXTENSION,2025,Finalizada,Hibrida,1S,16.0,NaN,SI,Género; Inclusión,10.0,EXTENSIÓN ACADÉMICA 2025.xlsx,2025
1,2537,Difusión carrera de administración pública- UTEM,FCJS,Administración_Pública,EXTENSION,2025,Finalizada,Presencial,1S,1.0,NaN,SI,Responsabilidad social; Inclusión,27.0,EXTENSIÓN ACADÉMICA 2025.xlsx,2025
2,2409,"Entre líneas: Procesos, relevos y traspasos en...",FCCOT,Arquitectura,EXTENSION,2025,Finalizada,Presencial,1S,1.0,NaN,SI,Género,39.0,EXTENSIÓN ACADÉMICA 2025.xlsx,2025


## 5. Exportacion a `data/clean/` (CSV + Parquet)

In [7]:
CLEAN.mkdir(parents=True, exist_ok=True)
salidas = {"dashboard_participantes_2024_2025": tabla_A,
           "dashboard_historico_2022_2025": tabla_B}
for nombre, t in salidas.items():
    t.to_csv(CLEAN / f"{nombre}.csv", index=False, encoding="utf-8")
    t.to_parquet(CLEAN / f"{nombre}.parquet", index=False)
    print(f"Escrito {nombre}: CSV + Parquet ({t.shape[0]} filas)")

Escrito dashboard_participantes_2024_2025: CSV + Parquet (521 filas)
Escrito dashboard_historico_2022_2025: CSV + Parquet (826 filas)


## 6. Reporte de cobertura por columna (% poblado por anio)

In [8]:
def cobertura(t):
    rep = {}
    for c in t.columns:
        if c in ("anio", "_anio"):
            continue
        rep[c] = t.groupby("anio")[c].apply(lambda s: round(100 * is_populated(s).mean(), 1))
    return pd.DataFrame(rep).T


print("=== COBERTURA Tabla A - dashboard_participantes_2024_2025 ===\n")
print(cobertura(tabla_A).to_string())
print("\n=== COBERTURA Tabla B - dashboard_historico_2022_2025 ===\n")
print(cobertura(tabla_B).to_string())

=== COBERTURA Tabla A - dashboard_participantes_2024_2025 ===

anio                        2024   2025
codigo                      96.0   87.0
nombre_iniciativa          100.0  100.0
facultad                    98.3  100.0
carrera                     98.9  100.0
instrumento                100.0  100.0
estado_sisav                98.3   87.0
modalidad                   98.9   99.4
semestre                    98.9  100.0
cantidad_act_planificadas   97.7   98.3
cantidad_act_ejecutadas      0.0    0.0
evidencia                   96.0  100.0
n_estudiantes               98.9   99.1
n_academicos_docentes       98.3   99.4
n_titulados                 97.2   99.1
n_organizaciones_osc        95.5   80.9
competencia_sello           98.9   99.4
competencia_generica        94.3  100.0
dominios_disciplinares      88.1   88.4
catedra_asignatura          56.8   58.6
_archivo_origen            100.0  100.0

=== COBERTURA Tabla B - dashboard_historico_2022_2025 ===

anio                        2022   20

## 7. Verificacion: numero de iniciativas por anio coherente con el diagnostico

El notebook 06 conto (tras dedup, universo de iniciativas reales, excluyendo el
archivo sin anio): 2022=148, 2023=157, 2024=176, 2025=345.

In [9]:
conteo_A = tabla_A["anio"].value_counts().sort_index().to_dict()
conteo_B = tabla_B["anio"].value_counts().sort_index().to_dict()
print("Tabla A por anio:", conteo_A)
print("Tabla B por anio:", conteo_B)

esperado = {2022: 148, 2023: 157, 2024: 176, 2025: 345}
assert conteo_B == esperado, f"MISMATCH vs diagnostico: {conteo_B} != {esperado}"
assert conteo_A == {2024: 176, 2025: 345}
print("\nOK: cifras coherentes con el diagnostico (notebook 06).")

Tabla A por anio: {2024: 176, 2025: 345}
Tabla B por anio: {2022: 148, 2023: 157, 2024: 176, 2025: 345}

OK: cifras coherentes con el diagnostico (notebook 06).


## 8. Notas para el equipo de visualizacion

- El diccionario completo esta en `docs/diccionario_dashboard.md`: describe cada
  columna, cuales son multivalor, desde que anio y que tabla usar para que grafico.
- **Participantes (Tabla A) -> metricas de participantes**. **Evolucion del numero
  de iniciativas (Tabla B) -> graficos de serie temporal.**
- **Advertencia importante:** no sumar las columnas de participantes de la Tabla A
  esperando totales institucionales. La cobertura es *por iniciativa registrada*
  (no es un censo); una suma subestima el total real y mezcla iniciativas con y
  sin dato. Usar promedios/medianas por iniciativa o totales con su cobertura
  declarada, no como cifra institucional cerrada.
- **Emprendedores / empleadores:** ausente en todas las fuentes. No se incluye.
  Si se requiere, hay que levantarlo con VcM porque el dato no se captura.